In [18]:
from google.colab import drive
import os
import torch
from torch.utils.data import Dataset, DataLoader
drive.mount('/content/drive')
import cv2
from google.colab.patches import cv2_imshow
from torch.utils.data import DataLoader
import numpy as np
import torchvision.transforms as transforms
import timm
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
ROOT = "/content/drive/MyDrive/Digital_Knee_X_ray_Images"
TRAIN_DATASET_DIR = "MedicalExpert-I"
VALIDATE_DATASET_DIR = "MedicalExpert-II"

In [20]:
class KneeXRayDataset(Dataset):

    def load_images_path(self):
        #iter each category
        # print(self.categories)
        for i, category in enumerate(self.categories):
            category_path = os.path.join(self.root, category)

            #iter file in category
            for file_path in os.listdir(category_path):
                self.image_paths.append(os.path.join(category_path, file_path))
                self.labels.append(i)

    def load_image_from_path(self,imagePath):
        img_bgr = cv2.imread(imagePath)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        return img_rgb
        
    def __init__(self, root, train_dataset_dir, validate_dataset_dir, transform , train=True, ):
        self.root = root
        self.transform  = transform

        # determine which dir will use (train , val)
        if train :
            self.root = os.path.join(root, train_dataset_dir)
        else:
            self.root = os.path.join(root,validate_dataset_dir)

        # get all category (0 -> 4)
        self.categories = os.listdir(self.root)
        self.image_paths = []
        self.labels = []

        # load images from dataset
        self.load_images_path()

        
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        image = self.load_image_from_path(self.image_paths[idx])
        # cv2_imshow(image)
        # cv2.waitKey(0)
        # cv2.destroyAllWindows()
        return self.transform(image), self.labels[idx]



In [21]:
import cv2

#function too add padding to image
class SquarePadOpenCV(object):
    def __call__(self, image):
        h, w = image.shape[:2]
        max_wh = max(h, w)
        pad_top = (max_wh - h) // 2
        pad_bottom = max_wh - h - pad_top
        pad_left = (max_wh - w) // 2
        pad_right = max_wh - w - pad_left
        
        padded_image = cv2.copyMakeBorder(
            image, 
            pad_top, pad_bottom, pad_left, pad_right, 
            borderType=cv2.BORDER_CONSTANT, 
            value=[0, 0, 0]
        )
        return padded_image


In [22]:
transform = transforms.Compose([
    SquarePadOpenCV(),
    transforms.ToTensor(),
    transforms.Resize((224,224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [23]:
train_dataset = KneeXRayDataset(root=ROOT, train_dataset_dir=TRAIN_DATASET_DIR, validate_dataset_dir=VALIDATE_DATASET_DIR, transform=transform)
validate_dataset = KneeXRayDataset(root=ROOT, train_dataset_dir=TRAIN_DATASET_DIR, validate_dataset_dir=VALIDATE_DATASET_DIR, train=False , transform=transform)

In [24]:
train_dataset_loader =  DataLoader(train_dataset, batch_size=16, shuffle=True , drop_last=False)
validate_dataset_loader = DataLoader(validate_dataset, batch_size=16, drop_last=False)

In [25]:
#load avaiable device 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [26]:
#define transfer learning model function
def create_transfer_learning_model(model_name: str="efficientnet_b0", pretrained: bool = True, num_classes:int = 5, device = device):
    model = timm.create_model(model_name, pretrained= pretrained, num_classes=5)

    # #frezze all other layer
    # for param in model.parameters():
    #     param.requires_grad = False

    # #unfrezze the class classifier10
    # for param in model.classifier.parameters():
    #     param.requires_grad = True

    #unlock all layer
    for param in model.parameters():
        param.requires_grad = True

    return model.to(device)


efficientnet_b0_model = create_transfer_learning_model()

In [27]:
# loss function & optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(filter(lambda p: p.requires_grad, efficientnet_b0_model.parameters()), lr=1e-5)


In [31]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    # model is tranining mode
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc=" Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        # delete previous grad
        optimizer.zero_grad()
        outputs = model(images)

        # calc loss
        loss = criterion(outputs, labels)

        # backward to calc loss
        loss.backward()

        # optimize model
        optimizer.step()

        # result
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total , 100 * correct / total

    

    

In [34]:
def validate(model, loader, criterion, device):

    # eval mode
    model.eval()

    # init
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=" Validating", leave=False):

            #load to device
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / total , 100 * correct / total 

    

In [36]:
EPOCHS = 15
best_val_acc = 0.0
save_path = '/content/drive/MyDrive/AI/models/EffiicentNetB0/best_efficientnet_b0_transfer.pth'
parent_dir = os.path.dirname(save_path)
os.makedirs(parent_dir, exist_ok=True)


for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(efficientnet_b0_model, train_dataset_loader, criterion, optimizer, device)

    print(f"  -> Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

    val_loss, val_acc = validate(efficientnet_b0_model, validate_dataset_loader, criterion, device)
    print(f"  -> Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        
        torch.save(efficientnet_b0_model.state_dict(), save_path)

        print(f"  [Save] Best val acc: {best_val_acc:.2f}%")
        


  -> Train Loss: 1.2077 | Train Acc: 60.18%


  -> Val Loss: 0.8707 | Val Acc: 70.24%
  [Save] Best val acc: 70.24%


  -> Train Loss: 0.9741 | Train Acc: 65.76%


  -> Val Loss: 0.7324 | Val Acc: 73.82%
  [Save] Best val acc: 73.82%


  -> Train Loss: 0.8012 | Train Acc: 71.39%


...

  -> Val Loss: 0.0900 | Val Acc: 98.36%
  [Save] Best val acc: 98.36%


  -> Train Loss: 0.1997 | Train Acc: 93.39%


  -> Val Loss: 0.0817 | Val Acc: 98.67%
  [Save] Best val acc: 98.67%


  -> Train Loss: 0.1794 | Train Acc: 94.97%


  -> Val Loss: 0.0817 | Val Acc: 98.67%


In [45]:
def predict_single_image_opencv(image_path, model, device):
    img_bgr = cv2.imread(image_path)
    
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    img_tensor = transform(img_rgb)
    
    img_batch = img_tensor.unsqueeze(0)
    
    img_batch = img_batch.to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(img_batch)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, dim=1)
        
    categories = ['0Normal', '1Doubtful', '2Mild', '3Moderate', '4Severe']
    return categories[predicted_idx.item()], confidence.item() * 100


In [51]:
def load_model_from_weight_file(model, path, device):
    checkpoint = torch.load(path, map_location=device)

    model.load_state_dict(checkpoint)

    model.eval()

    return model.to(device)

In [52]:
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=5)
model = load_model_from_weight_file(model, save_path, device)


In [54]:
image_test = "/content/drive/MyDrive/Digital_Knee_X_ray_Images/MedicalExpert-II/4Severe/SevereG4 (206).png"
predicted_class, confidence_score = predict_single_image_opencv(image_test, model, device)
print(predicted_class)
print(confidence_score)


4Severe
98.2745349407196


In [ ]:
# =====================================================================
# 1. INSTALL AND CONFIGURE NEST_ASYNCIO FOR JUPYTER/COLAB ENVIRONMENT
# =====================================================================
# Install required libraries directly in the notebook
%pip install fastapi uvicorn python-multipart nest_asyncio

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import threading
import urllib.request
import os

# Allow uvicorn to run nested within the Jupyter event loop
nest_asyncio.apply()

# Initialize FastAPI application
app = FastAPI(title="Knee OA Jupyter API")

# Define request schema (expects JSON containing the local image path)
class ImagePathRequest(BaseModel):
    image_path: str

# Define prediction endpoint
@app.post("/predict")
def predict_from_path(request: ImagePathRequest):
    path = request.image_path
    
    # Check if the file exists at the specified path
    if not os.path.exists(path):
        raise HTTPException(status_code=400, detail=f"Image not found at path: {path}")
        
    try:
        # Call the prediction function defined in your previous cells
        predicted_class, confidence_score = predict_single_image_opencv(
            request.image_path, model, device
        )
        
        return {
            "success": True,
            "image_path": request.image_path,
            "prediction": predicted_class,
            "confidence": f"{confidence_score:.2f}%"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Inference error: {str(e)}")

# Function to run the uvicorn server
def start_server():
    # Run server on all interfaces (0.0.0.0) at port 8000
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Start uvicorn server in a separate background thread
server_thread = threading.Thread(target=start_server, name="fastapi_server_thread")
server_thread.daemon = True
server_thread.start()

print("\n--- FASTAPI SERVER STARTED IN BACKGROUND ---")

# =====================================================================
# 2. AUTOMATICALLY RETRIEVE COLAB PUBLIC IP (LOCALTUNNEL PASSWORD)
# =====================================================================
try:
    # Fetch the public IP address of the Google Colab VM
    colab_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print("\n[Localtunnel Password] Copy this IP to bypass localtunnel security check:")
    print(f"===> {colab_ip} <===")
except Exception as e:
    print("\n[Warning] Could not fetch IP automatically. Please run: !curl ipv4.icanhazip.com")
    
print("\nNext step: Run the next cell to get your public API URL.")
